In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.models import Model

2025-02-06 21:47:11.868608: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-06 21:47:11.883929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738907231.902662 3961238 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738907231.908011 3961238 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-06 21:47:11.927313: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
coords = np.load('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0_points.npy')
print(coords.shape)

(9000, 17, 2)


In [5]:
non_missing_mask = coords != (0, 0)
print(non_missing_mask.shape)
non_missing_mask.sum()

(9000, 17, 2)


np.int64(291016)

In [7]:
non_missing_mask_tensor = tf.constant(non_missing_mask, dtype=tf.bool)
def non_missing_loss(y_true, y_pred):
    y_true_masked = tf.boolean_mask(y_true, non_missing_mask_tensor)
    y_pred_masked = tf.boolean_mask(y_pred, non_missing_mask_tensor)
    return tf.keras.losses.MeanSquaredError()(y_true_masked, y_pred_masked)

In [10]:
import os

In [11]:
os.mkdir('results')

In [21]:
def masked_mse_loss(y_true, y_pred):
    """
    Compute mean squared error (MSE) only for valid points.
    A point is considered missing if its ground truth is (0, 0).
    """
    # Create a mask: valid points get 1; missing points (0,0) get 0.
    is_missing = tf.logical_and(tf.equal(y_true[..., 0], 0.0),
                                tf.equal(y_true[..., 1], 0.0))
    mask = tf.cast(tf.logical_not(is_missing), tf.float32)  # Shape: (batch, seq_len, 17)
    
    # Compute squared error per coordinate pair.
    squared_error = tf.square(y_true - y_pred)  # Shape: (batch, seq_len, 17, 2)
    # Sum errors over the two coordinates.
    squared_error = tf.reduce_sum(squared_error, axis=-1)  # Shape: (batch, seq_len, 17)
    
    # Zero-out errors for missing points.
    masked_squared_error = squared_error * mask
    
    # Average only over valid (non-missing) points.
    total_valid = tf.reduce_sum(mask) + K.epsilon()
    mse = tf.reduce_sum(masked_squared_error) / total_valid
    return mse

## m00

In [24]:
# Assuming coords is a NumPy array of shape (9000, 17, 2)
# Normalize coordinates to [0, 1] range
coords_normalized = coords[:2500].astype('float32') / [170, 174]

# Reshape to (9000, 34)
coords_flat = coords_normalized.reshape((2500, -1))

# Create sliding windows of frames
window_size = 5  # Adjust based on experimentation
X = []
for i in range(coords_flat.shape[0]):
    start = max(0, i - window_size // 2)
    end = min(coords_flat.shape[0], i + window_size // 2 + 1)
    pad_before = max(0, window_size // 2 - i)
    pad_after = max(0, (i + window_size // 2 + 1) - coords_flat.shape[0])
    window = coords_flat[start:end]
    if pad_before > 0 or pad_after > 0:
        window = np.pad(window, ((pad_before, pad_after), (0, 0)), mode='constant')
    X.append(window)
X = np.array(X)

# Split into train and validation (maintain temporal order)
train_size = int(0.8 * len(X))
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = X_train, X_val  # Autoencoder target is the input

# Build LSTM Autoencoder
input_seq = Input(shape=(window_size, 34))
x = LSTM(64, activation='relu', return_sequences=True)(input_seq)
x = LSTM(32, activation='relu', return_sequences=False)(x)
x = RepeatVector(window_size)(x)
x = LSTM(32, activation='relu', return_sequences=True)(x)
x = LSTM(64, activation='relu', return_sequences=True)(x)
output = TimeDistributed(Dense(34, activation='sigmoid'))(x)

model = Model(input_seq, output)
model.compile(optimizer='adam', loss=masked_mse_loss)

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Denoise the entire dataset
denoised_windows = model.predict(X_copy)
denoised_coords = denoised_windows[:, window_size // 2, :]  # Extract middle frame

# Reshape back to original format and unnormalize
denoised_coords = denoised_coords.reshape((-1, 17, 2))
denoised_coords = denoised_coords * [170, 174]  # Scale back to original coordinates

Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 87ms/step - loss: 1.0998 - val_loss: 0.2273
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0337 - val_loss: 0.1228
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0148 - val_loss: 0.0900
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0126 - val_loss: 0.0934
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0095 - val_loss: 0.1000
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0082 - val_loss: 0.0968
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0078 - val_loss: 0.0949
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0073 - val_loss: 0.0879
Epoch 9/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0071 - val_loss: 0.0864
Epoch 10/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0066 - val_loss: 0.0880
Epoch 11/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0058 - val_loss: 0.0869
Epoch 12/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0061 

In [25]:
denoised_coords.shape

(9000, 17, 2)

In [26]:
np.save('results/denoised_coords_m00.npy', denoised_coords)

## m0

In [19]:
# Assuming coords is a NumPy array of shape (9000, 17, 2)
# Normalize coordinates to [0, 1] range
coords_normalized = coords.astype('float32') / [170, 174]

# Reshape to (9000, 34)
coords_flat = coords_normalized.reshape((9000, -1))

# Create sliding windows of frames
window_size = 5  # Adjust based on experimentation
X = []
for i in range(coords_flat.shape[0]):
    start = max(0, i - window_size // 2)
    end = min(coords_flat.shape[0], i + window_size // 2 + 1)
    pad_before = max(0, window_size // 2 - i)
    pad_after = max(0, (i + window_size // 2 + 1) - coords_flat.shape[0])
    window = coords_flat[start:end]
    if pad_before > 0 or pad_after > 0:
        window = np.pad(window, ((pad_before, pad_after), (0, 0)), mode='constant')
    X.append(window)
X = np.array(X)

# Split into train and validation (maintain temporal order)
train_size = int(0.8 * len(X))
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = X_train, X_val  # Autoencoder target is the input

# Build LSTM Autoencoder
input_seq = Input(shape=(window_size, 34))
x = LSTM(64, activation='relu', return_sequences=True)(input_seq)
x = LSTM(32, activation='relu', return_sequences=False)(x)
x = RepeatVector(window_size)(x)
x = LSTM(32, activation='relu', return_sequences=True)(x)
x = LSTM(64, activation='relu', return_sequences=True)(x)
output = TimeDistributed(Dense(34, activation='sigmoid'))(x)

model = Model(input_seq, output)
model.compile(optimizer='adam', loss=masked_mse_loss)

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Denoise the entire dataset
denoised_windows = model.predict(X)
denoised_coords = denoised_windows[:, window_size // 2, :]  # Extract middle frame

# Reshape back to original format and unnormalize
denoised_coords = denoised_coords.reshape((-1, 17, 2))
denoised_coords = denoised_coords * [170, 174]  # Scale back to original coordinates

Epoch 1/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 0.9315 - val_loss: 1.4913
Epoch 2/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2772 - val_loss: 1.4320
Epoch 3/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2168 - val_loss: 1.3444
Epoch 4/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1841 - val_loss: 1.2014
Epoch 5/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1652 - val_loss: 1.1677
Epoch 6/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1443 - val_loss: 1.1402
Epoch 7/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1299 - val_loss: 1.1134
Epoch 8/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1180 - val_loss: 1.0896
Epoch 9/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1088 - val_loss: 1.0556
Epoch 10/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1003 - val_loss: 1.0597
Epoch 11/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0959 - val_loss: 1.0392
Epoch 12/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/ste

In [22]:
X_copy = X.copy()

In [20]:
np.save('results/denoised_coords_m0.npy', denoised_coords)

## m1

In [8]:
# Assuming coords is a NumPy array of shape (9000, 17, 2)
# Normalize coordinates to [0, 1] range
coords_normalized = coords.astype('float32') / [170, 174]

# Reshape to (9000, 34)
coords_flat = coords_normalized.reshape((9000, -1))

# Create sliding windows of frames
window_size = 5  # Adjust based on experimentation
X = []
for i in range(coords_flat.shape[0]):
    start = max(0, i - window_size // 2)
    end = min(coords_flat.shape[0], i + window_size // 2 + 1)
    pad_before = max(0, window_size // 2 - i)
    pad_after = max(0, (i + window_size // 2 + 1) - coords_flat.shape[0])
    window = coords_flat[start:end]
    if pad_before > 0 or pad_after > 0:
        window = np.pad(window, ((pad_before, pad_after), (0, 0)), mode='constant')
    X.append(window)
X = np.array(X)

# Split into train and validation (maintain temporal order)
train_size = int(0.8 * len(X))
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = X_train, X_val  # Autoencoder target is the input

# Build LSTM Autoencoder
input_seq = Input(shape=(window_size, 34))
x = LSTM(64, activation='relu', return_sequences=True)(input_seq)
x = LSTM(32, activation='relu', return_sequences=False)(x)
x = RepeatVector(window_size)(x)
x = LSTM(32, activation='relu', return_sequences=True)(x)
x = LSTM(64, activation='relu', return_sequences=True)(x)
output = TimeDistributed(Dense(34, activation='sigmoid'))(x)

model = Model(input_seq, output)
model.compile(optimizer='adam', loss=non_missing_loss)

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Denoise the entire dataset
denoised_windows = model.predict(X)
denoised_coords = denoised_windows[:, window_size // 2, :]  # Extract middle frame

# Reshape back to original format and unnormalize
denoised_coords = denoised_coords.reshape((-1, 17, 2))
denoised_coords = denoised_coords * [170, 174]  # Scale back to original coordinates

Epoch 1/50


ValueError: Shapes (32, 5, 34) and (9000, 17, 2) are incompatible

In [12]:
np.save('results/denoised_coords_m1.npy', denoised_coords)

## m2

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# =============================================================================
# 1. Data Preparation: Creating Temporal Sequences
# =============================================================================
# Assume your original tracked points are stored in a NumPy array of shape:
#    (num_frames, 17, 2)
# For this example, we simulate dummy data:
num_frames = 9000
tracked_points = coords.copy()

# We choose a sequence (window) length; e.g., 10 consecutive frames.
sequence_length = 10

# Create overlapping sequences so that the model can learn from temporal context.
# Each sequence has shape (sequence_length, 17, 2).
def create_sequences(data, seq_length):
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

sequences = create_sequences(tracked_points, sequence_length)
print("Sequences shape:", sequences.shape)
# Expected shape: (num_frames - sequence_length + 1, sequence_length, 17, 2)

# =============================================================================
# 2. Building the Temporal Autoencoder Model
# =============================================================================
# We will design an encoder-decoder architecture that:
#   - Flattens each frame's spatial dimensions (17x2 -> 34 features)
#   - Uses an LSTM encoder to capture temporal dynamics and compress the sequence
#   - Uses a RepeatVector and LSTM decoder to reconstruct the sequence

# Define the input shape for one sequence.
input_shape = (sequence_length, 17, 2)
inputs = layers.Input(shape=input_shape, name='input_sequence')

# -- Preprocessing --
# Apply TimeDistributed Flatten so that each frame becomes a 34-dimensional vector.
x = layers.TimeDistributed(layers.Flatten(), name='flatten_per_frame')(inputs)  # shape: (T, 34)

# Optionally add Gaussian noise for a denoising effect.
x = layers.GaussianNoise(0.05, name='gaussian_noise')(x)

# -- Encoder --
# Process the sequence with an LSTM. We set return_sequences=False so that the
# LSTM outputs a single latent vector representing the entire sequence.
latent = layers.LSTM(64, activation='relu', name='encoder_lstm')(x)
# 'latent' now has shape (batch_size, 64)

# -- Decoder --
# Repeat the latent vector for each time step.
x_decoded = layers.RepeatVector(sequence_length, name='repeat_vector')(latent)

# Use an LSTM decoder that returns a sequence.
x_decoded = layers.LSTM(64, activation='relu', return_sequences=True, name='decoder_lstm')(x_decoded)

# Map back to the flattened dimension using TimeDistributed Dense layers.
x_decoded = layers.TimeDistributed(layers.Dense(34), name='time_distributed_dense')(x_decoded)

# Reshape each time step back to the original (17, 2) configuration.
outputs = layers.TimeDistributed(layers.Reshape((17, 2)), name='output_sequence')(x_decoded)

# Define and compile the autoencoder model.
autoencoder = models.Model(inputs, outputs, name='temporal_autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# =============================================================================
# 3. Training the Temporal Autoencoder
# =============================================================================
# In an unsupervised fashion, we train the network to reconstruct its own input.
# This forces the network to learn a temporally consistent manifold of points.
history = autoencoder.fit(
    sequences, sequences,
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

# =============================================================================
# 4. Using the Autoencoder for Improved Point Estimation
# =============================================================================
# After training, you can pass a sequence of tracked points through the autoencoder
# to obtain a temporally smoothed version. For example:
improved_sequences = autoencoder.predict(sequences)

# Note:
#   - improved_sequences has the same shape as the input sequences:
#         (num_sequences, sequence_length, 17, 2)
#   - To recover an improved track for each individual frame, you can use strategies
#     such as averaging overlapping predictions.


Sequences shape: (8991, 10, 17, 2)


Model: "temporal_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequence (InputLayer)     │ (None, 10, 17, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_per_frame               │ (None, 10, 34)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise (GaussianNoise)  │ (None, 10, 34)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_lstm (LSTM)             │ (None, 64)             │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_lstm (LSTM)             │ (None, 10, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_dense          │ (None, 10, 34)         │         2,210 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_sequence                 │ (None, 10, 17, 2)      │             0 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,578 (236.63 KB)

 Trainable params: 60,578 (236.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - loss: 4660.5210 - val_loss: 1963.4222
Epoch 2/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 612.1135 - val_loss: 1792.8167
Epoch 3/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 475.3500 - val_loss: 1594.9725
Epoch 4/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 421.2015 - val_loss: 1495.0878
Epoch 5/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 388.4399 - val_loss: 1417.3867
Epoch 6/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 349.7978 - val_loss: 1336.9647
Epoch 7/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 324.2769 - val_loss: 1315.4261
Epoch 8/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 303.8669 - val_loss: 1285.0503
Epoch 9/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 301.3470 - val_loss: 1278.0482
Epoch 10/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 287.4003 - val_loss: 1340.8751
Epoch 11/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 282.2924 - val_loss: 1279

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# =============================================================================
# 1. Data Preparation: Creating Temporal Sequences
# =============================================================================
# Assume your original tracked points are stored in a NumPy array of shape:
#    (num_frames, 17, 2)
# Here we simulate some dummy data.
num_frames = 9000
# For demonstration, random values in [0,1) are generated.
# In your real data, missing points will be exactly (0,0).
tracked_points = np.random.rand(num_frames, 17, 2).astype(np.float32)

# Simulate some missing points by setting ~10% of points to (0,0)
missing_fraction = 0.1
mask_missing = np.random.rand(num_frames, 17) < missing_fraction
tracked_points[mask_missing, 0] = 0.0
tracked_points[mask_missing, 1] = 0.0

# We choose a sequence (window) length; e.g., 10 consecutive frames.
sequence_length = 10

# Create overlapping sequences so that the model can learn from temporal context.
# Each sequence has shape (sequence_length, 17, 2).
def create_sequences(data, seq_length):
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

sequences = create_sequences(tracked_points, sequence_length)
print("Sequences shape:", sequences.shape)
# Expected shape: (num_frames - sequence_length + 1, sequence_length, 17, 2)

# =============================================================================
# 2. Custom Loss Function: Masked MSE
# =============================================================================
def masked_mse_loss(y_true, y_pred):
    """
    Compute mean squared error (MSE) only for valid points.
    A point is considered invalid if its ground truth is (0, 0).
    """
    # Create a mask where valid points have value 1 and missing points (0,0) have 0.
    # We check that BOTH x and y are zero to consider a point missing.
    is_missing = tf.logical_and(tf.equal(y_true[..., 0], 0.0),
                                tf.equal(y_true[..., 1], 0.0))
    mask = tf.cast(tf.logical_not(is_missing), tf.float32)  # Shape: (batch, seq_len, 17)

    # Compute squared error per coordinate pair.
    # This yields a tensor of shape: (batch, seq_len, 17, 2)
    squared_error = tf.square(y_true - y_pred)
    
    # Sum the error for each point across the two coordinates, resulting in shape (batch, seq_len, 17)
    squared_error = tf.reduce_sum(squared_error, axis=-1)
    
    # Apply the mask: errors for missing points become 0.
    masked_squared_error = squared_error * mask
    
    # Compute the mean over only the valid points.
    total_valid = tf.reduce_sum(mask) + K.epsilon()
    mse = tf.reduce_sum(masked_squared_error) / total_valid
    return mse

# =============================================================================
# 3. Building the Temporal Autoencoder Model
# =============================================================================
# Define the input shape for one sequence.
input_shape = (sequence_length, 17, 2)
inputs = layers.Input(shape=input_shape, name='input_sequence')

# -- Preprocessing --
# Use TimeDistributed to flatten each frame's spatial dimensions (17x2 -> 34 features).
x = layers.TimeDistributed(layers.Flatten(), name='flatten_per_frame')(inputs)  # shape: (T, 34)

# Optionally add Gaussian noise for a denoising effect.
x = layers.GaussianNoise(0.05, name='gaussian_noise')(x)

# -- Encoder --
# Process the sequence with an LSTM. Setting return_sequences=False means the LSTM
# outputs a single latent vector representing the entire sequence.
latent = layers.LSTM(64, activation='relu', name='encoder_lstm')(x)
# 'latent' has shape (batch_size, 64)

# -- Decoder --
# Repeat the latent vector for each time step.
x_decoded = layers.RepeatVector(sequence_length, name='repeat_vector')(latent)

# Use an LSTM decoder that returns a sequence.
x_decoded = layers.LSTM(64, activation='relu', return_sequences=True, name='decoder_lstm')(x_decoded)

# Map back to the flattened dimension (34) using a TimeDistributed Dense layer.
x_decoded = layers.TimeDistributed(layers.Dense(34), name='time_distributed_dense')(x_decoded)

# Reshape each time step back to the original (17, 2) configuration.
outputs = layers.TimeDistributed(layers.Reshape((17, 2)), name='output_sequence')(x_decoded)

# Define and compile the autoencoder model using the custom masked loss.
autoencoder = models.Model(inputs, outputs, name='temporal_autoencoder')
autoencoder.compile(optimizer='adam', loss=masked_mse_loss)
autoencoder.summary()

# =============================================================================
# 4. Training the Temporal Autoencoder
# =============================================================================
# Train the autoencoder to reconstruct its input (ignoring missing points).
history = autoencoder.fit(
    sequences, sequences,
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

# =============================================================================
# 5. Using the Autoencoder for Improved Point Estimation
# =============================================================================
# After training, pass a sequence of tracked points through the autoencoder
# to obtain a temporally smoothed version.
improved_sequences = autoencoder.predict(sequences)

# Note:
#   - improved_sequences has the same shape as the input sequences:
#         (num_sequences, sequence_length, 17, 2)
#   - Since sequences overlap, you might average the overlapping predictions for a
#     final per-frame result.


In [13]:
np.save('results/denoised_coords_m2.npy', improved_sequences)

## m3

In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# =============================================================================
# 1. Data Preparation: Creating Temporal Sequences
# =============================================================================
# Assume your tracked points are originally in pixel coordinates with shape:
#    (num_frames, 17, 2)
# The canvas size is (170, 174), so valid coordinates range roughly between 0 and
# 170 (vertical) and 0 and 174 (horizontal). Missing points are represented as (0,0).

num_frames = 9000
# Here we simulate some dummy data (in pixel coordinates)
tracked_points = coords.copy()

# Choose a sequence length (e.g., 10 consecutive frames)
sequence_length = 5

# Create overlapping sequences so that the model can learn temporal context.
# Each sequence has shape: (sequence_length, 17, 2)
def create_sequences(data, seq_length):
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

sequences = create_sequences(tracked_points, sequence_length)
print("Sequences shape:", sequences.shape)
# Expected shape: (num_frames - sequence_length + 1, sequence_length, 17, 2)

# =============================================================================
# 2. Define Normalization Factors and Custom Loss Function
# =============================================================================
# Normalization factors based on the canvas size:
# - For the first coordinate (assumed vertical): divide by 170.
# - For the second coordinate (assumed horizontal): divide by 174.
norm_factors = tf.constant([170.0, 174.0])

def masked_mse_loss(y_true, y_pred):
    """
    Compute mean squared error (MSE) only for valid points.
    A point is considered missing if its ground truth is (0, 0).
    """
    # Create a mask: valid points get 1; missing points (0,0) get 0.
    is_missing = tf.logical_and(tf.equal(y_true[..., 0], 0.0),
                                tf.equal(y_true[..., 1], 0.0))
    mask = tf.cast(tf.logical_not(is_missing), tf.float32)  # Shape: (batch, seq_len, 17)
    
    # Compute squared error per coordinate pair.
    squared_error = tf.square(y_true - y_pred)  # Shape: (batch, seq_len, 17, 2)
    # Sum errors over the two coordinates.
    squared_error = tf.reduce_sum(squared_error, axis=-1)  # Shape: (batch, seq_len, 17)
    
    # Zero-out errors for missing points.
    masked_squared_error = squared_error * mask
    
    # Average only over valid (non-missing) points.
    total_valid = tf.reduce_sum(mask) + K.epsilon()
    mse = tf.reduce_sum(masked_squared_error) / total_valid
    return mse

# =============================================================================
# 3. Building the Temporal Autoencoder Model with Normalization Layers
# =============================================================================
input_shape = (sequence_length, 17, 2)
inputs = layers.Input(shape=input_shape, name='input_sequence')

# --- Normalization ---
# Normalize the input coordinates from pixel space to [0,1].
# Each point (y, x) is divided by (170, 174) respectively.
norm_inputs = layers.Lambda(lambda x: x / norm_factors, name='normalize')(inputs)

# --- Preprocessing ---
# Flatten each frame's 17x2 coordinates into a 34-dimensional vector.
x = layers.TimeDistributed(layers.Flatten(), name='flatten_per_frame')(norm_inputs)

# Optionally add Gaussian noise for a denoising effect.
x = layers.GaussianNoise(0.05, name='gaussian_noise')(x)

# --- Encoder ---
# Use an LSTM to capture the temporal dynamics and compress the sequence.
latent = layers.LSTM(64, activation='relu', name='encoder_lstm')(x)
# 'latent' has shape (batch_size, 64)

# --- Decoder ---
# Repeat the latent vector for each time step.
x_decoded = layers.RepeatVector(sequence_length, name='repeat_vector')(latent)
# Decode the latent vector into a sequence.
x_decoded = layers.LSTM(64, activation='relu', return_sequences=True, name='decoder_lstm')(x_decoded)
# Map back to 34 features using a TimeDistributed Dense layer.
x_decoded = layers.TimeDistributed(layers.Dense(34), name='time_distributed_dense')(x_decoded)
# Reshape each time step back to (17, 2).
outputs = layers.TimeDistributed(layers.Reshape((17, 2)), name='output_sequence')(x_decoded)

# --- De-normalization ---
# Scale the output back from [0,1] to pixel coordinates.
outputs_scaled = layers.Lambda(lambda x: x * norm_factors, name='denormalize')(outputs)

# Build and compile the model.
autoencoder = models.Model(inputs, outputs_scaled, name='temporal_autoencoder')
autoencoder.compile(optimizer='adam', loss=masked_mse_loss)
autoencoder.summary()

# =============================================================================
# 4. Training the Temporal Autoencoder
# =============================================================================
# The network is trained to reconstruct the original (pixel-scale) sequences.
# Note: The input sequences are in pixel coordinates, and normalization + de-normalization
# are performed inside the model.
history = autoencoder.fit(
    sequences, sequences,  # Both inputs and targets are in pixel space.
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

# =============================================================================
# 5. Using the Autoencoder for Improved Point Estimation
# =============================================================================
# After training, pass sequences through the model to get improved (denoised) coordinates.
# The output will be automatically scaled back to pixel coordinates.
improved_sequences = autoencoder.predict(sequences)

# You can then combine overlapping predictions (e.g., by averaging) to obtain per-frame estimates.


Sequences shape: (8996, 5, 17, 2)


Model: "temporal_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequence (InputLayer)     │ (None, 5, 17, 2)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalize (Lambda)              │ (None, 5, 17, 2)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_per_frame               │ (None, 5, 34)          │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise (GaussianNoise)  │ (None, 5, 34)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_lstm (LSTM)             │ (None, 64)             │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_lstm (LSTM)             │ (None, 5, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_dense          │ (None, 5, 34)          │         2,210 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_sequence                 │ (None, 5, 17, 2)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ denormalize (Lambda)            │ (None, 5, 17, 2)       │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,578 (236.63 KB)

 Trainable params: 60,578 (236.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - loss: 4358.6875 - val_loss: 1153.5410
Epoch 2/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 158.8255 - val_loss: 947.5050
Epoch 3/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 111.1966 - val_loss: 834.0157
Epoch 4/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 91.8146 - val_loss: 812.6630
Epoch 5/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 81.6713 - val_loss: 735.6451
Epoch 6/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 72.7307 - val_loss: 703.2491
Epoch 7/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 64.4992 - val_loss: 679.0994
Epoch 8/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 61.1660 - val_loss: 692.8355
Epoch 9/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 57.0083 - val_loss: 690.6882
Epoch 10/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 53.2942 - val_loss: 680.8387
Epoch 11/50
253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 48.9542 - val_loss: 646.6077
Epoch 12/50
2

In [15]:
np.save('results/denoised_coords_m3.npy', improved_sequences)

## m4

In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# =============================================================================
# 1. Data Preparation: Creating Temporal Sequences
# =============================================================================
# Assume your tracked points are originally in pixel coordinates with shape:
#    (num_frames, 17, 2)
# The canvas size is (170, 174), so valid coordinates range roughly between 0 and
# 170 (vertical) and 0 and 174 (horizontal). Missing points are represented as (0,0).

num_frames = 9000
# Here we simulate some dummy data (in pixel coordinates)
tracked_points = coords.copy()

# Choose a sequence length (e.g., 10 consecutive frames)
sequence_length = 5

# Create overlapping sequences so that the model can learn temporal context.
# Each sequence has shape: (sequence_length, 17, 2)
def create_sequences(data, seq_length):
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

subsequences = create_sequences(tracked_points[:2500], sequence_length)
sequences = create_sequences(tracked_points, sequence_length)
print("Sequences shape:", sequences.shape)
# Expected shape: (num_frames - sequence_length + 1, sequence_length, 17, 2)

# =============================================================================
# 2. Define Normalization Factors and Custom Loss Function
# =============================================================================
# Normalization factors based on the canvas size:
# - For the first coordinate (assumed vertical): divide by 170.
# - For the second coordinate (assumed horizontal): divide by 174.
norm_factors = tf.constant([170.0, 174.0])

def masked_mse_loss(y_true, y_pred):
    """
    Compute mean squared error (MSE) only for valid points.
    A point is considered missing if its ground truth is (0, 0).
    """
    # Create a mask: valid points get 1; missing points (0,0) get 0.
    is_missing = tf.logical_and(tf.equal(y_true[..., 0], 0.0),
                                tf.equal(y_true[..., 1], 0.0))
    mask = tf.cast(tf.logical_not(is_missing), tf.float32)  # Shape: (batch, seq_len, 17)
    
    # Compute squared error per coordinate pair.
    squared_error = tf.square(y_true - y_pred)  # Shape: (batch, seq_len, 17, 2)
    # Sum errors over the two coordinates.
    squared_error = tf.reduce_sum(squared_error, axis=-1)  # Shape: (batch, seq_len, 17)
    
    # Zero-out errors for missing points.
    masked_squared_error = squared_error * mask
    
    # Average only over valid (non-missing) points.
    total_valid = tf.reduce_sum(mask) + K.epsilon()
    mse = tf.reduce_sum(masked_squared_error) / total_valid
    return mse

# =============================================================================
# 3. Building the Temporal Autoencoder Model with Normalization Layers
# =============================================================================
input_shape = (sequence_length, 17, 2)
inputs = layers.Input(shape=input_shape, name='input_sequence')

# --- Normalization ---
# Normalize the input coordinates from pixel space to [0,1].
# Each point (y, x) is divided by (170, 174) respectively.
norm_inputs = layers.Lambda(lambda x: x / norm_factors, name='normalize')(inputs)

# --- Preprocessing ---
# Flatten each frame's 17x2 coordinates into a 34-dimensional vector.
x = layers.TimeDistributed(layers.Flatten(), name='flatten_per_frame')(norm_inputs)

# Optionally add Gaussian noise for a denoising effect.
x = layers.GaussianNoise(0.05, name='gaussian_noise')(x)

# --- Encoder ---
# Use an LSTM to capture the temporal dynamics and compress the sequence.
latent = layers.LSTM(64, activation='relu', name='encoder_lstm')(x)
# 'latent' has shape (batch_size, 64)

# --- Decoder ---
# Repeat the latent vector for each time step.
x_decoded = layers.RepeatVector(sequence_length, name='repeat_vector')(latent)
# Decode the latent vector into a sequence.
x_decoded = layers.LSTM(64, activation='relu', return_sequences=True, name='decoder_lstm')(x_decoded)
# Map back to 34 features using a TimeDistributed Dense layer.
x_decoded = layers.TimeDistributed(layers.Dense(34), name='time_distributed_dense')(x_decoded)
# Reshape each time step back to (17, 2).
outputs = layers.TimeDistributed(layers.Reshape((17, 2)), name='output_sequence')(x_decoded)

# --- De-normalization ---
# Scale the output back from [0,1] to pixel coordinates.
outputs_scaled = layers.Lambda(lambda x: x * norm_factors, name='denormalize')(outputs)

# Build and compile the model.
autoencoder = models.Model(inputs, outputs_scaled, name='temporal_autoencoder')
autoencoder.compile(optimizer='adam', loss=masked_mse_loss)
autoencoder.summary()

# =============================================================================
# 4. Training the Temporal Autoencoder
# =============================================================================
# The network is trained to reconstruct the original (pixel-scale) sequences.
# Note: The input sequences are in pixel coordinates, and normalization + de-normalization
# are performed inside the model.
history = autoencoder.fit(
    subsequences, subsequences,  # Both inputs and targets are in pixel space.
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

# =============================================================================
# 5. Using the Autoencoder for Improved Point Estimation
# =============================================================================
# After training, pass sequences through the model to get improved (denoised) coordinates.
# The output will be automatically scaled back to pixel coordinates.
improved_sequences = autoencoder.predict(sequences)

# You can then combine overlapping predictions (e.g., by averaging) to obtain per-frame estimates.

Sequences shape: (8996, 5, 17, 2)


Model: "temporal_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequence (InputLayer)     │ (None, 5, 17, 2)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalize (Lambda)              │ (None, 5, 17, 2)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_per_frame               │ (None, 5, 34)          │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise (GaussianNoise)  │ (None, 5, 34)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_lstm (LSTM)             │ (None, 64)             │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_lstm (LSTM)             │ (None, 5, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_dense          │ (None, 5, 34)          │         2,210 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_sequence                 │ (None, 5, 17, 2)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ denormalize (Lambda)            │ (None, 5, 17, 2)       │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,578 (236.63 KB)

 Trainable params: 60,578 (236.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 10s 74ms/step - loss: 9272.1777 - val_loss: 545.3309
Epoch 2/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 134.9988 - val_loss: 224.8010
Epoch 3/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 53.3917 - val_loss: 162.9064
Epoch 4/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 44.2828 - val_loss: 123.4551
Epoch 5/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 35.7801 - val_loss: 95.7060
Epoch 6/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 31.1952 - val_loss: 84.8876
Epoch 7/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 29.2698 - val_loss: 76.5272
Epoch 8/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 29.0245 - val_loss: 79.0212
Epoch 9/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 28.0758 - val_loss: 78.2990
Epoch 10/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 26.9928 - val_loss: 76.3294
Epoch 11/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 27.0965 - val_loss: 67.8809
Epoch 12/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 1

In [18]:
np.save('results/denoised_coords_m4.npy', improved_sequences)